# RAGAS Synthetic Test Data Generation from Uploaded PDFs

This notebook generates synthetic RAG test data from user-uploaded PDF files using the LangChain document loading path only.

Flow:

- Upload one or more PDF files.
- Load them with LangChain `PyPDFLoader`.
- Split them with `RecursiveCharacterTextSplitter`.
- Generate a synthetic RAGAS test set.
- Display and export the compact dataset.

## 1. Install dependencies

Run once per session. On Colab, restart the runtime after installing before continuing.

In [ ]:
%pip install \
    ragas==0.3.9 \
    langchain==0.3.27 \
    langchain-openai==0.3.28 \
    langchain-community==0.3.27 \
    langchain-huggingface==0.1.2 \
    openai==1.99.5 \
    datasets==4.0.0 \
    pandas==2.2.3 \
    sentence-transformers==3.4.1 \
    rapidfuzz==3.10.1 \
    pypdf==5.9.0
print('Install complete. If running on Colab, restart the runtime before proceeding.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.7/366.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 16.4 MB/

Install complete. If running on Colab, restart the runtime before proceeding.


## 2. Imports, configuration, and model setup

Set credentials and all reusable generation settings here only.

In [ ]:
import os
from pathlib import Path
from typing import Iterable

from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import AzureChatOpenAI

from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms.base import LangchainLLMWrapper
from ragas.testset import TestsetGenerator

# --- Azure OpenAI credentials and model settings ---
AZURE_OPENAI_API_KEY = "FkrroqaFXJ3w3AAABACOGKkmt"
AZURE_OPENAI_ENDPOINT = "https:/enai.azure.com/"
AZURE_OPENAI_API_VERSION = "202view"
AZURE_OPENAI_DEPLOYMENT = "gpt-5"  # Your Azure GPT-5 deployment name
AZURE_OPENAI_MODEL_NAME = "gpt-5"  # Example: "gpt-5"

# --- Local embedding and generation settings ---
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TESTSET_SIZE = 5
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100
UPLOAD_DIR = Path("uploaded_pdfs")

# Keep only the columns needed by your downstream evaluation/reporting.
# Update this list after inspecting generated_df.columns for your RAGAS version.
KEEP_COLUMNS = [
    "user_input",
    "reference",
    "reference_contexts",
    "synthesizer_name",
]

os.environ["AZURE_OPENAI_API_KEY"] = AZURE_OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = AZURE_OPENAI_ENDPOINT
os.environ["AZURE_OPENAI_API_VERSION"] = AZURE_OPENAI_API_VERSION
UPLOAD_DIR.mkdir(exist_ok=True)


# GPT-5 rejects any temperature other than 1. RAGAS otherwise forces a low
# temperature on the LLM (0.01, or 0.3 when n > 1), so pin temperature=1.
class GPT5LangchainLLMWrapper(LangchainLLMWrapper):
    def generate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
        return super().generate_text(prompt, n=n, temperature=1, stop=stop, callbacks=callbacks)

    async def agenerate_text(self, prompt, n=1, temperature=None, stop=None, callbacks=None):
        return await super().agenerate_text(prompt, n=n, temperature=1, stop=stop, callbacks=callbacks)


def compact_testset_dataframe(testset, keep_columns: list[str] = KEEP_COLUMNS):
    generated_df = testset.to_pandas()
    available_columns = [column for column in keep_columns if column in generated_df.columns]
    compact_df = generated_df[available_columns].copy()

    print("All generated columns:")
    print(list(generated_df.columns))
    return compact_df


# --- LangChain clients for RAGAS test generation ---
langchain_azure_llm = AzureChatOpenAI(
    openai_api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT.strip(),
    azure_deployment=AZURE_OPENAI_DEPLOYMENT,
    model=AZURE_OPENAI_MODEL_NAME,
    temperature=1,
    validate_base_url=False,
)
langchain_generator_llm = GPT5LangchainLLMWrapper(langchain_azure_llm, bypass_temperature=True)

langchain_hf_embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
)
langchain_generator_embeddings = LangchainEmbeddingsWrapper(langchain_hf_embeddings)

print("Configuration and LangChain model clients initialized.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Configuration and LangChain model clients initialized.


/tmp/ipykernel_945/1753172397.py:79: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  langchain_generator_embeddings = LangchainEmbeddingsWrapper(langchain_hf_embeddings)


## 3. Upload source PDF documents

Upload one or more PDF files. These uploaded PDFs are the only source material used for both generation paths.

In [ ]:
def upload_pdf_files(upload_dir: Path = UPLOAD_DIR) -> list[Path]:
    """Upload any number of PDFs in Colab, or use local PDF_PATHS outside Colab."""
    try:
        from google.colab import files  # type: ignore
    except ImportError:
        pdf_paths = [Path(p) for p in globals().get("PDF_PATHS", [])]
        pdf_paths = [p for p in pdf_paths if p.exists() and p.suffix.lower() == ".pdf"]
        if not pdf_paths:
            raise RuntimeError(
                "This notebook is not running in Colab. Set PDF_PATHS to a list of local PDF files, then rerun this cell."
            )
        return pdf_paths

    uploaded = files.upload()  # Select one or many PDFs in the file picker.
    pdf_paths: list[Path] = []

    for filename, content in uploaded.items():
        if not filename.lower().endswith(".pdf"):
            print(f"Skipping non-PDF file: {filename}")
            continue

        output_path = upload_dir / Path(filename).name
        output_path.write_bytes(content)
        pdf_paths.append(output_path)

    if not pdf_paths:
        raise ValueError("No PDF files were uploaded. Please upload at least one .pdf file.")

    return pdf_paths


pdf_paths = upload_pdf_files()
print(f"Uploaded/selected {len(pdf_paths)} PDF file(s).")
for pdf_path in pdf_paths:
    print(f"- {pdf_path.name}")

Saving HR_Leave_Policy.pdf to HR_Leave_Policy.pdf
Uploaded/selected 1 PDF file(s).
- HR_Leave_Policy.pdf


## 4. LangChain test data generation

Load the PDFs as LangChain documents, split them, and generate a RAGAS test set.

In [ ]:
def load_langchain_pdf_documents(pdf_paths: Iterable[Path]) -> list[Document]:
    """Load uploaded PDFs into LangChain documents, preserving source metadata."""
    documents: list[Document] = []

    for pdf_path in pdf_paths:
        loader = PyPDFLoader(str(pdf_path))
        pages = loader.load()

        for page in pages:
            page.page_content = page.page_content.replace("\x00", " ").strip()
            page.metadata.update({"source": pdf_path.name, "source_path": str(pdf_path)})

        documents.extend(page for page in pages if page.page_content)

    if not documents:
        raise ValueError("The uploaded PDFs did not contain extractable text. Try OCR first for scanned PDFs.")

    return documents


langchain_docs = load_langchain_pdf_documents(pdf_paths)

langchain_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
langchain_split_docs = langchain_text_splitter.split_documents(langchain_docs)

print(f"Loaded {len(langchain_docs)} page document(s).")
print(f"Split them into {len(langchain_split_docs)} LangChain chunk(s).")
print()
print("First LangChain chunk preview:")
print(langchain_split_docs[0].page_content[:800])

Loaded 8 page document(s).
Split them into 19 LangChain chunk(s).

First LangChain chunk preview:
MERIDIAN TECHNOLOGIES PVT. LTD.
CONFIDENTIAL
Page 1
 MERIDIAN TECHNOLOGIES PVT. LTD.
 Internal Policy Document
Leave Policy
Document Number: HR-POL-001 | Version: 2.1 | Effective Date: January 2024
Classification: Internal / Confidential
This document is the property of Meridian Technologies Pvt. Ltd. Unauthorized reproduction is prohibited.


In [ ]:
langchain_generator = TestsetGenerator(
    llm=langchain_generator_llm,
    embedding_model=langchain_generator_embeddings,
)

langchain_dataset = langchain_generator.generate_with_langchain_docs(
    langchain_split_docs,
    testset_size=TESTSET_SIZE,
)

Applying SummaryExtractor:   0%|          | 0/12 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/19 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/12 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/19 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/6 [00:00<?, ?it/s]

### LangChain generated test set

In [ ]:
langchain_compact_df = compact_testset_dataframe(langchain_dataset)
langchain_compact_df

All generated columns:
['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']


,user_input,reference,reference_contexts,synthesizer_name
0,Meridian Technologies Pvt. Ltd. Leave Policy d...,Document Number: HR-POL-001; Version: 2.1; Eff...,[MERIDIAN TECHNOLOGIES PVT. LTD.\nCONFIDENTIAL...,single_hop_specific_query_synthesizer
1,"In MERIDIAN TECHNOLOGIES PVT. LTD., which sect...",Section 8: Leave Application Procedure.,[MERIDIAN TECHNOLOGIES PVT. LTD.\nCONFIDENTIAL...,single_hop_specific_query_synthesizer
2,For a probationer with 3 months of continuous ...,Probationers accrue EL but cannot avail it unt...,[<1-hop>\n\nMERIDIAN TECHNOLOGIES PVT. LTD.\nC...,multi_hop_abstract_query_synthesizer
3,"Confirmd emp: EL/SL/CL entitlments per yr, min...",- Annual entitlements (confirmed employees): 1...,"[<1-hop>\n\n(SL), Casual Leave (CL), Maternity...",multi_hop_abstract_query_synthesizer
4,Cn i club Sick Leave with Earned Leave 4 a vac...,Sick Leave cannot be clubbed with Earned Leave...,[<1-hop>\n\nthe year for rest and recovery.\n4...,multi_hop_specific_query_synthesizer
5,HR policy clarification (web-search style): Fo...,Encashment of Sick Leave is not permitted unde...,[<1-hop>\n\nEncashment is at the basic salary ...,multi_hop_specific_query_synthesizer


## 5. Export generated dataset

RAGAS creates its standard test-set schema during generation. To keep only the needed columns, export the compact DataFrame.

In [ ]:
from IPython.display import display

langchain_compact_df.to_csv("ragas_langchain_generated_testset.csv", index=False)

print("Saved:")
print("- ragas_langchain_generated_testset.csv")

print("\nLangChain generated dataset:")
display(langchain_compact_df)

Saved:
- ragas_langchain_generated_testset.csv

LangChain generated dataset:


,user_input,reference,reference_contexts,synthesizer_name
0,Meridian Technologies Pvt. Ltd. Leave Policy d...,Document Number: HR-POL-001; Version: 2.1; Eff...,[MERIDIAN TECHNOLOGIES PVT. LTD.\nCONFIDENTIAL...,single_hop_specific_query_synthesizer
1,"In MERIDIAN TECHNOLOGIES PVT. LTD., which sect...",Section 8: Leave Application Procedure.,[MERIDIAN TECHNOLOGIES PVT. LTD.\nCONFIDENTIAL...,single_hop_specific_query_synthesizer
2,For a probationer with 3 months of continuous ...,Probationers accrue EL but cannot avail it unt...,[<1-hop>\n\nMERIDIAN TECHNOLOGIES PVT. LTD.\nC...,multi_hop_abstract_query_synthesizer
3,"Confirmd emp: EL/SL/CL entitlments per yr, min...",- Annual entitlements (confirmed employees): 1...,"[<1-hop>\n\n(SL), Casual Leave (CL), Maternity...",multi_hop_abstract_query_synthesizer
4,Cn i club Sick Leave with Earned Leave 4 a vac...,Sick Leave cannot be clubbed with Earned Leave...,[<1-hop>\n\nthe year for rest and recovery.\n4...,multi_hop_specific_query_synthesizer
5,HR policy clarification (web-search style): Fo...,Encashment of Sick Leave is not permitted unde...,[<1-hop>\n\nEncashment is at the basic salary ...,multi_hop_specific_query_synthesizer
